In [24]:
import os
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build


In [25]:
# Change only this URL, then run all cells top-to-bottom
SHEET_URL = 'https://docs.google.com/spreadsheets/d/1sdKQAXGeYuAH8IqG4fNm7YylmY-ydO95692ynTHgicY/edit?usp=sharing'

# Optional: set to a tab title if you want to force a specific tab
TARGET_TAB_NAME = None

CREDENTIALS_PATH = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS', 'auto_auth.json')
HEADER_SCAN_ROWS = 10
SPIKE_Q3_QUANTILE = 0.90
GREEN_BG = {'red': 0.85, 'green': 0.97, 'blue': 0.85}

REQUIRED_COLUMN_ALIASES = {
    'video_link': ['video_link', 'video link', 'video_url', 'video url', 'link', 'url'],
    'upload_date': ['upload_date', 'upload date', 'date', 'posted_date', 'post_date', 'created_at'],
    'views': ['views', 'view', 'play_count', 'play count'],
    'saves': ['saves', 'save', 'bookmarks', 'favorites', 'favourites'],
}

def extract_spreadsheet_id(url: str) -> str:
    m = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', str(url))
    if not m:
        raise ValueError(f'Invalid Google Sheets URL: {url}')
    return m.group(1)

def normalize_header(value) -> str:
    return re.sub(r'[^a-z0-9]+', '_', str(value or '').strip().lower()).strip('_')

def make_unique_headers(raw_headers, width):
    out = []
    seen = {}
    for i in range(width):
        h = str(raw_headers[i]).strip() if i < len(raw_headers) else ''
        if not h:
            h = f'column_{i + 1}'
        base = h
        n = seen.get(base, 0) + 1
        seen[base] = n
        if n > 1:
            h = f'{base}_{n}'
        out.append(h)
    return out

def normalize_link(value) -> str:
    s = str(value or '').strip()
    if not s:
        return ''
    return s.split('?', 1)[0].rstrip('/')

def first_match_key_in_row(row, aliases):
    row_keys = {normalize_header(c) for c in row}
    for alias in aliases:
        if normalize_header(alias) in row_keys:
            return True
    return False

def detect_header_row(sample_rows, required_aliases):
    for r_idx, row in enumerate(sample_rows):
        ok = True
        for aliases in required_aliases.values():
            if not first_match_key_in_row(row, aliases):
                ok = False
                break
        if ok:
            return r_idx
    return None

def pick_column_name(df_columns, aliases):
    normalized_to_actual = {}
    for c in df_columns:
        key = normalize_header(c)
        if key and key not in normalized_to_actual:
            normalized_to_actual[key] = c
    for alias in aliases:
        key = normalize_header(alias)
        if key in normalized_to_actual:
            return normalized_to_actual[key]
    return None

if not os.path.isfile(CREDENTIALS_PATH):
    raise FileNotFoundError(
        f'Credentials not found: {CREDENTIALS_PATH}. '
        'Set GOOGLE_APPLICATION_CREDENTIALS or place auto_auth.json in this folder.'
    )

spreadsheet_id = extract_spreadsheet_id(SHEET_URL)
creds = Credentials.from_service_account_file(
    CREDENTIALS_PATH,
    scopes=['https://www.googleapis.com/auth/spreadsheets'],
)
service = build('sheets', 'v4', credentials=creds)

print(f'Sheet configured: {spreadsheet_id}')


Sheet configured: 1sdKQAXGeYuAH8IqG4fNm7YylmY-ydO95692ynTHgicY


In [26]:
# Load source data from the configured Google Sheet
meta = service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
sheet_candidates = []

for sheet in meta.get('sheets', []):
    props = sheet.get('properties', {})
    title = props.get('title', '')
    sheet_id = props.get('sheetId')
    if not title or sheet_id is None:
        continue
    if TARGET_TAB_NAME and title != TARGET_TAB_NAME:
        continue

    safe_title = title.replace("'", "''")
    sample_range = f"'{safe_title}'!A1:ZZ{HEADER_SCAN_ROWS}"
    sample_rows = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=sample_range,
    ).execute().get('values', [])

    header_row_idx = detect_header_row(sample_rows, REQUIRED_COLUMN_ALIASES)
    if header_row_idx is not None:
        sheet_candidates.append((title, sheet_id, header_row_idx))

if not sheet_candidates:
    tab_names = [s.get('properties', {}).get('title', '') for s in meta.get('sheets', [])]
    raise ValueError(
        'Could not find a tab with required columns (video_link/upload_date/views/saves). '
        f'Tabs found: {tab_names}'
    )

selected_tab_title, selected_tab_id, header_row_idx = sheet_candidates[0]
if TARGET_TAB_NAME and selected_tab_title != TARGET_TAB_NAME:
    raise ValueError(f'TARGET_TAB_NAME={TARGET_TAB_NAME} was not matched.')

safe_title = selected_tab_title.replace("'", "''")
header_row_1_based = header_row_idx + 1
all_rows = service.spreadsheets().values().get(
    spreadsheetId=spreadsheet_id,
    range=f"'{safe_title}'!A{header_row_1_based}:ZZ",
).execute().get('values', [])

if not all_rows:
    raise ValueError(f'Tab {selected_tab_title} appears to be empty from header row onward.')

raw_headers = all_rows[0]
raw_data_rows = all_rows[1:]

width = max(len(raw_headers), max((len(r) for r in raw_data_rows), default=0))
headers = make_unique_headers(raw_headers, width)
rows = []
for r in raw_data_rows:
    padded = list(r) + [''] * (width - len(r))
    rows.append(padded[:width])

raw_df = pd.DataFrame(rows, columns=headers)

column_map = {}
for canonical_name, aliases in REQUIRED_COLUMN_ALIASES.items():
    chosen = pick_column_name(raw_df.columns, aliases)
    column_map[canonical_name] = chosen

missing = [k for k, v in column_map.items() if not v]
if missing:
    raise ValueError(
        f'Missing required column(s): {missing}. '
        f'Found columns: {list(raw_df.columns)}'
    )

videos_df = raw_df.rename(columns={v: k for k, v in column_map.items()}).copy()
videos_df = videos_df[['video_link', 'upload_date', 'views', 'saves']]

print(f'Loaded tab: {selected_tab_title}')
print(f'Header row: {header_row_1_based}')
print(f'Rows loaded (excluding header): {len(videos_df)}')
videos_df.head()


Loaded tab: Sheet1
Header row: 1
Rows loaded (excluding header): 1066


,video_link,upload_date,views,saves
0,https://www.tiktok.com/@carlhoos_/video/761613...,3/12/2026,"4,800,000","15,900"
1,https://www.tiktok.com/@artthuroficial_/video/...,3/12/2026,"665,800","2,200"
2,https://www.tiktok.com/@carlosferiag/video/761...,3/12/2026,"481,100",796
3,https://www.tiktok.com/@jesaaelys/video/761612...,3/12/2026,"446,500",543
4,https://www.tiktok.com/@kachitaak4/video/76162...,3/12/2026,"234,400",785


In [27]:
# Clean core fields and keep rows needed for analysis
def to_number(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace(',', '', regex=False)
        .str.replace(r'[^0-9.-]', '', regex=True)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors='coerce').fillna(0)

analysis_df = videos_df.copy()
analysis_df['video_link'] = analysis_df['video_link'].astype(str).str.strip()
analysis_df['views'] = to_number(analysis_df['views']).astype(int)
analysis_df['saves'] = to_number(analysis_df['saves']).astype(int)
analysis_df['upload_date'] = pd.to_datetime(analysis_df['upload_date'], errors='coerce')
analysis_df = analysis_df.dropna(subset=['upload_date'])
analysis_df = analysis_df[analysis_df['video_link'] != '']
analysis_df = analysis_df.sort_values('upload_date').reset_index(drop=True)

print(f'Rows ready for analysis: {len(analysis_df)}')
analysis_df[['video_link', 'upload_date', 'views', 'saves']].head()


Rows ready for analysis: 1066


,video_link,upload_date,views,saves
0,https://www.tiktok.com/@shonci/video/760533824...,2026-02-10,6400000,38100
1,https://www.tiktok.com/@dxxxx076/video/7605442...,2026-02-11,8400000,25400
2,https://www.tiktok.com/@moca_e_sousa/video/760...,2026-02-12,10200000,51000
3,https://www.tiktok.com/@coronelvalen_/video/76...,2026-02-13,4600000,9100
4,https://www.tiktok.com/@davigalon_/video/76065...,2026-02-14,5600000,13500


In [28]:
# Plot views and saves over time with dual y-axes and video link in hover
plot_df = analysis_df.copy()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=plot_df['upload_date'],
        y=plot_df['views'],
        name='Views',
        yaxis='y',
        line=dict(color='blue'),
        customdata=plot_df[['video_link']],
        hovertemplate='Upload Date: %{x|%Y-%m-%d}<br>Views: %{y:,}<br>Video: %{customdata[0]}<extra></extra>',
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df['upload_date'],
        y=plot_df['saves'],
        name='Saves',
        yaxis='y2',
        line=dict(color='red'),
        customdata=plot_df[['video_link']],
        hovertemplate='Upload Date: %{x|%Y-%m-%d}<br>Saves: %{y:,}<br>Video: %{customdata[0]}<extra></extra>',
    )
)

fig.update_layout(
    title=f'Views and Saves over Time ({selected_tab_title})',
    xaxis_title='Upload Date',
    yaxis=dict(title='Views', side='left'),
    yaxis2=dict(title='Saves', overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified',
    width=1200,
    height=600,
)

fig.show()


In [29]:
# Find spikes in views or saves (IQR-style rule using configurable upper quantile)
spike_df = analysis_df.copy()

views_q1, views_q3 = spike_df['views'].quantile([0.25, SPIKE_Q3_QUANTILE])
views_iqr = views_q3 - views_q1
views_threshold = views_q3 + 1.5 * views_iqr

saves_q1, saves_q3 = spike_df['saves'].quantile([0.25, SPIKE_Q3_QUANTILE])
saves_iqr = saves_q3 - saves_q1
saves_threshold = saves_q3 + 1.5 * saves_iqr

spike_df['views_spike'] = spike_df['views'] > views_threshold
spike_df['saves_spike'] = spike_df['saves'] > saves_threshold

spike_rows = spike_df.loc[
    spike_df['views_spike'] | spike_df['saves_spike'],
    ['upload_date', 'video_link', 'views', 'saves', 'views_spike', 'saves_spike'],
].copy()

spike_rows['spike_type'] = np.select(
    [
        spike_rows['views_spike'] & spike_rows['saves_spike'],
        spike_rows['views_spike'],
        spike_rows['saves_spike'],
    ],
    ['views_and_saves', 'views_only', 'saves_only'],
    default='none',
)

spike_rows = spike_rows.sort_values(['upload_date', 'views', 'saves'], ascending=[True, False, False])
spike_rows['upload_date'] = spike_rows['upload_date'].dt.strftime('%Y-%m-%d')

print(f'Views spike threshold: {views_threshold:,.0f}')
print(f'Saves spike threshold: {saves_threshold:,.0f}')

if spike_rows.empty:
    print('No spikes detected based on current thresholds.')
else:
    print('Dates and video links with spikes in views or saves:')
    print(spike_rows[['upload_date', 'video_link', 'spike_type', 'views', 'saves']].to_string(index=False))

spike_rows[['upload_date', 'video_link', 'spike_type', 'views', 'saves']]


Views spike threshold: 5,999,475
Saves spike threshold: 22,998
Dates and video links with spikes in views or saves:
upload_date                                                             video_link      spike_type    views  saves
 2026-02-10               https://www.tiktok.com/@shonci/video/7605338240983239954 views_and_saves  6400000  38100
 2026-02-11             https://www.tiktok.com/@dxxxx076/video/7605442038603009298 views_and_saves  8400000  25400
 2026-02-12         https://www.tiktok.com/@moca_e_sousa/video/7605998617312234784 views_and_saves 10200000  51000
 2026-02-14        https://www.tiktok.com/@sacul_e_sepol/video/7606668487804620064      views_only  6300000  12200
 2026-02-17         https://www.tiktok.com/@jei_am_music/video/7607908762229148935 views_and_saves 11600000  23700
 2026-02-18        https://www.tiktok.com/@lautypangallo/video/7608005295041694994      views_only  7200000  16800
 2026-02-19  https://www.tiktok.com/@rigbylokaporradeira/video/7608671281130523

,upload_date,video_link,spike_type,views,saves
0,2026-02-10,https://www.tiktok.com/@shonci/video/760533824...,views_and_saves,6400000,38100
1,2026-02-11,https://www.tiktok.com/@dxxxx076/video/7605442...,views_and_saves,8400000,25400
2,2026-02-12,https://www.tiktok.com/@moca_e_sousa/video/760...,views_and_saves,10200000,51000
5,2026-02-14,https://www.tiktok.com/@sacul_e_sepol/video/76...,views_only,6300000,12200
10,2026-02-17,https://www.tiktok.com/@jei_am_music/video/760...,views_and_saves,11600000,23700
12,2026-02-18,https://www.tiktok.com/@lautypangallo/video/76...,views_only,7200000,16800
21,2026-02-19,https://www.tiktok.com/@rigbylokaporradeira/vi...,views_and_saves,23200000,124700
18,2026-02-19,https://www.tiktok.com/@aaron.shm/video/760840...,saves_only,3400000,38400
23,2026-02-20,https://www.tiktok.com/@rigbylokaporradeira/vi...,views_and_saves,31500000,157700
24,2026-02-20,https://www.tiktok.com/@ocho4real8/video/76090...,views_and_saves,7900000,46600


In [30]:
# Mark rows in the configured Google Sheet green where video_link is in spike_rows
def contiguous_ranges(sorted_indices):
    if not sorted_indices:
        return []
    ranges = []
    start = prev = sorted_indices[0]
    for idx in sorted_indices[1:]:
        if idx == prev + 1:
            prev = idx
            continue
        ranges.append((start, prev + 1))
        start = prev = idx
    ranges.append((start, prev + 1))
    return ranges

if 'spike_rows' not in globals():
    raise RuntimeError('Run the spike detection cell first so spike_rows exists.')

spike_links = {normalize_link(v) for v in spike_rows['video_link'].dropna().tolist() if normalize_link(v)}
if not spike_links:
    raise RuntimeError('spike_rows has no video links to mark.')

meta = service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
total_marked = 0

for sheet in meta.get('sheets', []):
    props = sheet.get('properties', {})
    sheet_id = props.get('sheetId')
    title = props.get('title', '')
    col_count = int(props.get('gridProperties', {}).get('columnCount', 26))
    if sheet_id is None or not title:
        continue

    safe_title = title.replace("'", "''")
    sample_rows = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"'{safe_title}'!A1:ZZ{HEADER_SCAN_ROWS}",
    ).execute().get('values', [])

    header_row_idx = detect_header_row(sample_rows, {'video_link': REQUIRED_COLUMN_ALIASES['video_link']})
    if header_row_idx is None:
        print(f"[{title}] skipped: video_link header not found.")
        continue

    header_row_1_based = header_row_idx + 1
    header_values = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"'{safe_title}'!A{header_row_1_based}:ZZ{header_row_1_based}",
    ).execute().get('values', [[]])[0]

    temp_df = pd.DataFrame(columns=make_unique_headers(header_values, len(header_values)))
    link_col_name = pick_column_name(temp_df.columns, REQUIRED_COLUMN_ALIASES['video_link'])
    if not link_col_name:
        print(f"[{title}] skipped: could not map video_link column.")
        continue

    link_col_idx = list(temp_df.columns).index(link_col_name)
    data_start_row_1_based = header_row_1_based + 1
    data_rows = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"'{safe_title}'!A{data_start_row_1_based}:ZZ",
    ).execute().get('values', [])

    row_indices_to_mark = []
    for i, row in enumerate(data_rows):
        if link_col_idx >= len(row):
            continue
        if normalize_link(row[link_col_idx]) in spike_links:
            row_indices_to_mark.append((data_start_row_1_based - 1) + i)

    if not row_indices_to_mark:
        print(f"[{title}] no matching links found.")
        continue

    row_indices_to_mark = sorted(set(row_indices_to_mark))
    requests = []
    for start_row, end_row in contiguous_ranges(row_indices_to_mark):
        requests.append({
            'repeatCell': {
                'range': {
                    'sheetId': sheet_id,
                    'startRowIndex': start_row,
                    'endRowIndex': end_row,
                    'startColumnIndex': 0,
                    'endColumnIndex': col_count,
                },
                'cell': {'userEnteredFormat': {'backgroundColor': GREEN_BG}},
                'fields': 'userEnteredFormat.backgroundColor',
            }
        })

    service.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body={'requests': requests},
    ).execute()

    marked_count = len(row_indices_to_mark)
    total_marked += marked_count
    print(f"[{title}] marked {marked_count} row(s) green.")

print(f'Done. Total marked rows: {total_marked}')


[Sheet1] marked 45 row(s) green.
Done. Total marked rows: 45
